In [1]:
import re
import itertools
from itertools import chain
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import polars as pl

import statsmodels.formula.api as smf
import statsmodels.stats.anova as sa
from statsmodels.stats.multitest import fdrcorrection

from scipy.stats import mannwhitneyu

from novami.io.file import read_pl, write_pl
from novami.stats.utils import round_to_significant
from novami.stats.tests import *

# ChEMBL Deposition

In [81]:
df = read_pl("../data/ChEMBL/processed/chembl_measurements.xlsx")

## Mechanism / Readout missingness

In [82]:
frac_missing = df.filter(
    pl.any_horizontal(
        pl.col("assay_readout") == "AMB",
        pl.col("assay_mechanism") == "AMB",
        )
    )["assay_chembl_id"].n_unique() / df["assay_chembl_id"].n_unique()

print(f"Percent of not-annotated mechanisms/readouts: {frac_missing:.2%}")

Percent of not-annotated mechanisms/readouts: 33.26%


## ChEMBL deposition by year

In [58]:
df = read_pl("../data/ChEMBL/processed/chembl_measurements.xlsx")

In [63]:
total_counts = df.group_by("target_channel_name").agg(pl.col("molregno").n_unique().alias("Compounds"))

In [64]:
df = df.rename({"year":"Year"}).filter(pl.col("Year").is_not_null())

In [65]:
ddc = defaultdict(list)

min_year = df["Year"].min()
max_year = df["Year"].max()

year_range = list(np.arange(min_year, max_year + 1))
ddc["Year"] = year_range

for channel in df["target_channel_name"].unique().to_list():
    sub_df = df.filter(
        pl.col("target_channel_name") == channel
    )
    for year in year_range:
        year_df = sub_df.filter(
            pl.col("Year") <= year
        )
        n_unique = year_df["molregno"].n_unique()
        ddc[channel].append(
            n_unique
        )

In [66]:
count_df = pl.DataFrame(ddc)

In [67]:
df_long = count_df.unpivot(
    on=["Nav1.5", "hERG", "Kir2.1", "Cav1.2", "Kv7.1", "Kv4.3"],
    index="Year",
    variable_name="target_channel_name",
    value_name="Compounds"
)

In [70]:
write_pl(df_long, "../data/deposition/by_year.xlsx")
write_pl(total_counts, "../data/deposition/total.xlsx")

# Model and descriptor usage counts

In [146]:
study_df = read_pl('../data/supporting_tables/s1.xlsx')

In [147]:
time_breaks = [(2001, 2009), (2010, 2018), (2019, 2026)]

In [148]:
# Check the most frequently used descriptors

for period in time_breaks:
    print(f'Descriptors between {period[0]} and {period[1]}:')
    sub_df = study_df.filter(pl.col('Year').is_between(*period))
    dscs = sub_df['Features'].to_list()
    dscs_types = [re.sub(r'\([^)]*\)', '', item) for item in dscs]
    dscs_types = chain.from_iterable([item.split(',') for item in dscs_types])
    dscs_types = [item.strip() for item in dscs_types]

    ct = Counter(dscs_types)
    print('\t', ct.most_common(10))

Descriptors between 2001 and 2009:
	 [('MD', 19), ('3D Shape-based', 6), ('Fragment', 4), ('Pharmacophore', 3), ('Circular', 3), ('QC', 3), ('Field-based', 3), ('Fragment-based', 2), ('Topological', 2), ('Atom Types', 2)]
Descriptors between 2010 and 2018:
	 [('MD', 21), ('Circular', 13), ('Fragment', 10), ('Pharmacophore', 7), ('Path-based', 6), ('Field-based', 3), ('Topological', 3), ('4D-FP', 2), ('Atom Pairs', 2), ('custom SMARTS', 1)]
Descriptors between 2019 and 2026:
	 [('MD', 44), ('Circular', 40), ('Fragment', 29), ('Graphs', 22), ('Path-based', 11), ('Atom Pairs', 7), ('Embeddings', 6), ('Pharmacophore', 6), ('SMILES', 6), ('Embedding', 5)]


In [149]:
# Check the most frequently used models

for period in time_breaks:
    print(f'Models between {period[0]} and {period[1]}:')
    sub_df = study_df.filter(pl.col('Year').is_between(*period))
    models = sub_df['Models'].to_list()
    models = [re.sub(r'\([^)]*\)', '', item) for item in models]
    models = chain.from_iterable([item.split(',') for item in models])
    models = [item.strip() for item in models]
    ct = Counter(models)
    print('\t', ct.most_common(10))

Models between 2001 and 2009:
	 [('PLS', 9), ('SVM', 8), ('DT', 6), ('NB', 4), ('Alignment-based', 3), ('SOM', 3), ('MLR', 3), ('LR', 3), ('kNN', 3), ('NN', 2)]
Models between 2010 and 2018:
	 [('SVM', 15), ('RF', 14), ('PLS', 10), ('kNN', 8), ('NB', 5), ('DT', 4), ('LDA', 3), ('GBM', 3), ('GP', 2), ('MLR', 2)]
Models between 2019 and 2026:
	 [('RF', 37), ('SVM', 26), ('XGB', 18), ('kNN', 14), ('NB', 10), ('MLP', 10), ('DNN', 9), ('> DNN', 8), ('GCN', 7), ('GBM', 7)]


# Performance analysis

## Prepare relevant tables

In [191]:
df_cls = read_pl("../data/supporting_tables/s3.xlsx")
df_reg = read_pl("../data/supporting_tables/s4.xlsx")

In [192]:
df_cls = df_cls.with_columns(
    pl.col("Splitting/Evaluation").map_elements(
        lambda evl: re.sub(r"\\cite\{[\w\s,]+\}", "", evl).strip(),
        return_dtype=pl.String
    ).alias("EVL")
)

df_reg = df_reg.with_columns(
    pl.col("Splitting/Evaluation").map_elements(
        lambda evl: re.sub(r"\\cite\{[\w\s,]+\}", "", evl).strip(),
        return_dtype=pl.String
    ).alias("EVL")
)

In [193]:
all_evaluations = set(df_cls["EVL"].drop_nulls().unique().to_list()).union(set(df_reg["EVL"].drop_nulls().unique().to_list()))

In [194]:
evl_map = {
    'D-Optimal': "Uniform",
    'DISE': "Distance",
    'Distance': "Distance",
    'Diverse': "Uniform",
    'Diverse train' : "Uniform",
    'Ext' : "External",
    'Ext (Temporal)': "External",
    'External': "External",
    'In-house': None,
    'LCO CV': "LOO",
    'LOO CV': "LOO",
    'Manual': "Manual",
    'N/A': None,
    'Random' : "Random",
    'Random CV' :"Random",
    'Random CV: binding': "Random",
    'Random CV: clamp': "Random",
    'Random Nested CV': "Random",
    'Random: all assays': "Random",
    'Random: patch-clamp': "Random",
    'Scaffold': "Scaffold",
    'Scaffold CV': "Scaffold",
    'Scaffold Distance' : "Distance",
    'Sorted Uniform' : "Uniform",
    'Sorted Y-based': "Uniform",
    'Stratified CV' : "Stratified",
    'Stratified Random': "Stratified",
    'Stratified Random CV': "Stratified",
    'Temporal': "Temporal",
    None: None
}

In [195]:
df_cls = df_cls.with_columns(
    pl.col("EVL").map_elements(
        lambda evl: evl_map.get(evl, None),
        return_dtype=pl.String
    ).alias("Eval Class")
)

df_reg = df_reg.with_columns(
    pl.col("EVL").map_elements(
        lambda evl: evl_map.get(evl, None),
        return_dtype=pl.String
    ).alias("Eval Class")
)

In [196]:
df_cls = df_cls.with_columns(
    pl.col("Model").map_elements(
        lambda mod: re.search(r"([^\s]+)", mod).group(1),
        return_dtype=pl.String
    ).alias("Model Class")
)

df_reg = df_reg.with_columns(
    pl.col("Model").map_elements(
        lambda mod: re.search(r"([^\s]+)", mod).group(1),
        return_dtype=pl.String
    ).alias("Model Class")
)

In [199]:
write_pl(df_cls, "../data/performance/classification.xlsx")
write_pl(df_reg, "../data/performance/regression.xlsx")

## ANOVA analysis

In [200]:
class_df = read_pl("../data/performance/classification.xlsx")
reg_df = read_pl("../data/performance/regression.xlsx")

cdf = class_df.filter(
    pl.col("Threshold").is_in([
        "<10,>30", "10", "1", "<1,>10"
    ]),
    pl.col("Model Class").is_in([
        "ML", "DL"
    ]),
    pl.col("Eval Class").is_in([
        "Random", "External", "Distance"
    ]),
    pl.col("Channel") == "hERG"
)

rdf = reg_df.filter(
    pl.col("Model Class").is_in([
        "DL", "ML"
    ]),
    pl.col("Eval Class").is_in([
        "External", "Random", "Distance"
    ]),
    pl.col("Channel") == "hERG"
)

cdf = cdf.select(["Threshold", "ACC", "BA", "SEN", "SPE", "ROC AUC", "MCC", "Year", "Eval Class", "Model Class"])

rdf = rdf.select(["R2", "MAE", "RMSE", "Year", "Eval Class", "Model Class"])

cdf = cdf.unpivot(
    index=["Threshold", "Year", "Eval Class", "Model Class"],
    variable_name="Metric",
    value_name="Value",
)

rdf = rdf.unpivot(
    index=["Year", "Eval Class", "Model Class"],
    variable_name="Metric",
    value_name="Value",
)

cdf = cdf.filter(
    pl.col("Value").is_not_null()
)
rdf = rdf.filter(
    pl.col("Value").is_not_null()
)

cdf = cdf.rename({"Eval Class": "Eval", "Model Class": "Model"})
rdf = rdf.rename({"Eval Class": "Eval", "Model Class": "Model"})

In [201]:
cdf = cdf.with_columns(
    pl.when(
        pl.col("Year") <= 2009
    ).then(
        pl.lit("Foundation")
    ).when(
        pl.col("Year") <= 2018
    ).then(
        pl.lit("Consolidation")
    ).otherwise(
        pl.lit("Expansion")
    ).alias("Era")
)

rdf = rdf.with_columns(
    pl.when(
        pl.col("Year") <= 2009
    ).then(
        pl.lit("Foundation")
    ).when(
        pl.col("Year") <= 2018
    ).then(
        pl.lit("Consolidation")
    ).otherwise(
        pl.lit("Expansion")
    ).alias("Era")
)

In [202]:
write_pl(cdf, "../results/anova/cdf.xlsx")
write_pl(rdf, "../results/anova/rdf.xlsx")

In [203]:
cdf = cdf.to_pandas()

results = []

for metric in cdf["Metric"].unique():
    sub_df = cdf[cdf["Metric"] == metric]

    model = smf.ols(
        "Value ~ C(Model) + C(Eval) + C(Threshold) + C(Era)",
        data=sub_df
    ).fit()

    table = sa.anova_lm(model, typ=2)
    table["eta_squared"] = table["sum_sq"] / table["sum_sq"].sum()
    table.index = ["Model", "Eval", "Threshold", "Era", "Residual"]
    table.insert(0, "Metric", metric)
    results.append(table)

cres = pd.concat(results).sort_values(["Metric", "eta_squared"], ascending=[True, False])

In [204]:
rdf = rdf.to_pandas()

results = []

for metric in rdf["Metric"].unique():
    sub_df = rdf[rdf["Metric"] == metric]

    model = smf.ols(
        "Value ~ C(Model) + C(Eval) + C(Era)",
        data=sub_df
    ).fit()

    table = sa.anova_lm(model, typ=2)
    table["eta_squared"] = table["sum_sq"] / table["sum_sq"].sum()
    table.index = ["Model", "Eval", "Era", "Residual"]
    table.insert(0, "Metric", metric)
    results.append(table)

rres = pd.concat(results).sort_values(["Metric", "eta_squared"], ascending=[True, False])

In [205]:
write_pl(pl.from_pandas(cres), "../results/anova/classification.xlsx")
write_pl(pl.from_pandas(rres), "../results/anova/regression.xlsx")

## Mann-Whitney U test

In [7]:
cdf = read_pl("../results/anova/cdf.xlsx")
rdf = read_pl("../results/anova/rdf.xlsx")

In [9]:
def run_analysis(df: pl.DataFrame, pairs: list, min_n: int = 3, alpha: float = 0.05, alternative: str = "two-sided",):
    results = []

    for metric in df["Metric"].unique().to_list():
        mdf = df.filter(pl.col("Metric") == metric)
        rows = []

        for column, group_pairs in pairs:
            for label_1, label_2 in group_pairs:
                values_1 = mdf.filter(pl.col(column) == label_1)["Value"].to_numpy()
                values_2 = mdf.filter(pl.col(column) == label_2)["Value"].to_numpy()

                out = mann_whitney_u_test(
                    values_1=values_1,
                    values_2=values_2,
                    label_1=label_1,
                    label_2=label_2,
                    min_n=min_n,
                    alpha=alpha,
                    alternative=alternative,
                )

                out.update(
                    {
                        "Metric": metric,
                        "Comparison": column,
                    }
                )
                rows.append(out)

        if not rows:
            continue

        metric_df = pl.DataFrame(rows)

        valid = metric_df.filter(pl.col("p_value").is_not_null())
        invalid = metric_df.filter(pl.col("p_value").is_null()).with_columns(
            pl.lit(None, dtype=pl.Float64).alias("p_value_adj")
        )

        if valid.height > 0:
            _, p_adj = fdrcorrection(valid["p_value"].to_numpy(), alpha=alpha)
            valid = valid.with_columns(
                pl.Series("p_value_adj", p_adj, dtype=pl.Float64)
            )

        metric_df = pl.concat([valid, invalid], how="diagonal_relaxed")
        results.append(metric_df)

    if not results:
        return pl.DataFrame()

    combined = pl.concat(results, how="diagonal_relaxed").with_columns(
        pl.when(pl.col("p_value_adj").is_null())
        .then(None)
        .when(pl.col("p_value_adj") >= alpha)
        .then(pl.format("{} ~ {}", pl.col("label_1"), pl.col("label_2")))
        .when(pl.col("rbc") > 0)
        .then(pl.format("{} > {}", pl.col("label_1"), pl.col("label_2")))
        .when(pl.col("rbc") < 0)
        .then(pl.format("{} < {}", pl.col("label_1"), pl.col("label_2")))
        .otherwise(pl.format("{} ~ {}", pl.col("label_1"), pl.col("label_2")))
        .alias("outcome_adj")
    ).sort(["Metric", "p_value_adj"], nulls_last=True)

    return combined


In [10]:
pairs_cls = [
    ("Threshold", [*itertools.combinations(cdf["Threshold"].unique().to_list(), r=2)]),
    ("Eval", [*itertools.combinations(cdf["Eval"].unique().to_list(), r=2)]),
    ("Model", [*itertools.combinations(cdf["Model"].unique().to_list(), r=2)]),
    ("Era", [*itertools.combinations(cdf["Era"].unique().to_list(), r=2)])
]

pairs_reg = [
    ("Eval", [*itertools.combinations(rdf["Eval"].unique().to_list(), r=2)]),
    ("Model", [*itertools.combinations(rdf["Model"].unique().to_list(), r=2)]),
    ("Era", [*itertools.combinations(rdf["Era"].unique().to_list(), r=2)])
]

cls_df = run_analysis(cdf, pairs_cls)
reg_df = run_analysis(rdf, pairs_reg)

cls_df = cls_df.drop(["null_hypothesis", "mean_1", "mean_2", "u_stat", "outcome"])
reg_df = reg_df.drop(["null_hypothesis", "mean_1", "mean_2", "u_stat", "outcome"])

cls_df = cls_df.cast({
    "n1": pl.Int64,
    "n2": pl.Int64,
    "median_1": pl.Float64,
    "median_2": pl.Float64,
    "rbc": pl.Float64,
    "p_value_adj": pl.Float64,
    "p_value": pl.Float64,
})

reg_df = reg_df.cast({
    "n1": pl.Int64,
    "n2": pl.Int64,
    "median_1": pl.Float64,
    "median_2": pl.Float64,
    "rbc": pl.Float64,
    "p_value_adj": pl.Float64,
    "p_value": pl.Float64,
})


Sample size too small for <Expansion ~ Foundation>. N1: 62, N2: 0
Sample size too small for <Foundation ~ Consolidation>. N1: 0, N2: 9
Sample size too small for <<10,>30 ~ 1>. N1: 0, N2: 0
Sample size too small for <<10,>30 ~ 10>. N1: 0, N2: 19
Sample size too small for <<10,>30 ~ <1,>10>. N1: 0, N2: 1
Sample size too small for <1 ~ 10>. N1: 0, N2: 19
Sample size too small for <1 ~ <1,>10>. N1: 0, N2: 1
Sample size too small for <10 ~ <1,>10>. N1: 19, N2: 1
Sample size too small for <Random ~ Distance>. N1: 10, N2: 2
Sample size too small for <Distance ~ External>. N1: 2, N2: 8
Sample size too small for <Expansion ~ Foundation>. N1: 18, N2: 0
Sample size too small for <Expansion ~ Consolidation>. N1: 18, N2: 2
Sample size too small for <Foundation ~ Consolidation>. N1: 0, N2: 2
Sample size too small for <<10,>30 ~ <1,>10>. N1: 3, N2: 2
Sample size too small for <1 ~ <1,>10>. N1: 6, N2: 2
Sample size too small for <10 ~ <1,>10>. N1: 60, N2: 2
Sample size too small for <Expansion ~ Found

In [ ]:
write_pl(cls_df, "../results/mwu/classification.xlsx")
write_pl(reg_df, "../results/mwu/regression.xlsx")

In [11]:
def print_table(df: pl.DataFrame, task="cls"):

    if task == "cls":
        metrics = ["ACC", "BA", "SEN", "SPE", "ROC AUC", "MCC"]
        comps = ["Era", "Model", "Eval", "Threshold"]
    else:
        metrics = ["R2", "RMSE", "MAE"]
        comps = ["Era", "Model", "Eval"]

    for idx, comp in enumerate(comps):
        if idx != 0:
            print("\t" + r"\midrule")
        print("\t" + r"\multicolumn{12}{c}{" + f"{comp}" + "}" + r" \\")
        print("\t" + r"\midrule")

        edf = (
            df
            .filter(pl.col("Comparison") == comp)
            .drop("Comparison")
            .sort(["Metric", "p_value_adj", "p_value"], nulls_last=True)
        )

        for midx, mt in enumerate(metrics):
            mdf = edf.filter(pl.col("Metric") == mt)

            if mdf.height == 0:
                continue

            mt_line = "\t" + r"\multirow{" + f"{len(mdf)}" + "}{*}{" + f"{mt}" + "}"
            print(mt_line)

            mx_lab = max(len(str(item)) for item in (mdf["label_1"].to_list() + mdf["label_2"].to_list()))
            mx_n = max(len(str(item)) for item in (mdf["n1"].to_list() + mdf["n2"].to_list()))

            for row in mdf.iter_rows(named=True):
                g1, g2 = row["label_1"], row["label_2"]
                n1, n2 = row["n1"], row["n2"]
                m1, m2 = row["median_1"], row["median_2"]
                pv, pva = row["p_value"], row["p_value_adj"]
                rbc, e_size, low_flag = row["rbc"], row["effect_size"], row["low_n_flag"]

                if pv is not None:
                    pv = round_to_significant(pv, 2)
                    pva = round_to_significant(pva, 2) if pva is not None else None
                    m1 = round_to_significant(m1, 2)
                    m2 = round_to_significant(m2, 2)
                    outcome = row["outcome_adj"].replace("~", r"$\sim$")

                if pv is None:
                    row_line = (
                        "\t\t"
                        + f"& {g1:<{mx_lab}} & {n1:<{mx_n}} & - "
                        + f"& {g2:<{mx_lab}} & {n2:<{mx_n}} & - "
                        + f"& - & - & - & - & -"
                        + r" \\"
                    )
                else:
                    row_line = (
                        "\t\t"
                        + f"& {g1:<{mx_lab}} & {n1:<{mx_n}} & {m1:.2f} "
                        + f"& {g2:<{mx_lab}} & {n2:<{mx_n}} & {m2:.2f} "
                        + f"& {rbc:.2f} & {e_size} "
                        + f"& {pv} & {pva} "
                        + f"& {outcome}"
                        + r" \\"
                    )

                print(row_line)

            if midx != len(metrics) - 1:
                print("\t" + r"\cmidrule{2-12}")


In [12]:
print_table(cls_df, task="cls")

	\multicolumn{12}{c}{Era} \\
	\midrule
	\multirow{3}{*}{ACC}
		& Expansion     & 77 & 0.83 & Foundation    & 6  & 0.90 & -0.42 & Medium & 0.086 & 0.22 & Expansion $\sim$ Foundation \\
		& Foundation    & 6  & 0.90 & Consolidation & 17 & 0.85 & 0.32 & Medium & 0.26 & 0.52 & Foundation $\sim$ Consolidation \\
		& Expansion     & 77 & 0.83 & Consolidation & 17 & 0.85 & -0.06 & Negligible & 0.72 & 0.78 & Expansion $\sim$ Consolidation \\
	\cmidrule{2-12}
	\multirow{3}{*}{BA}
		& Expansion     & 18 & - & Foundation    & 0  & - & - & - & - & - & - \\
		& Expansion     & 18 & - & Consolidation & 2  & - & - & - & - & - & - \\
		& Foundation    & 0  & - & Consolidation & 2  & - & - & - & - & - & - \\
	\cmidrule{2-12}
	\multirow{3}{*}{SEN}
		& Expansion     & 75 & 0.83 & Foundation    & 8  & 0.86 & -0.15 & Small & 0.49 & 0.9 & Expansion $\sim$ Foundation \\
		& Foundation    & 8  & 0.86 & Consolidation & 16 & 0.85 & 0.13 & Small & 0.62 & 0.9 & Foundation $\sim$ Consolidation \\
		& Expansion    

In [13]:
print_table(reg_df, task="reg")

	\multicolumn{12}{c}{Era} \\
	\midrule
	\multirow{3}{*}{R2}
		& Expansion     & 16 & 0.62 & Foundation    & 5  & 0.85 & -0.65 & Large & 0.035 & 0.25 & Expansion $\sim$ Foundation \\
		& Consolidation & 3  & 0.68 & Foundation    & 5  & 0.85 & -0.47 & Medium & 0.37 & 0.87 & Consolidation $\sim$ Foundation \\
		& Expansion     & 16 & 0.62 & Consolidation & 3  & 0.68 & -0.31 & Medium & 0.43 & 0.87 & Expansion $\sim$ Consolidation \\
	\cmidrule{2-12}
	\multirow{3}{*}{RMSE}
		& Expansion     & 19 & 0.60 & Foundation    & 3  & 0.74 & -0.42 & Medium & 0.27 & 0.45 & Expansion $\sim$ Foundation \\
		& Expansion     & 19 & - & Consolidation & 1  & - & - & - & - & - & - \\
		& Consolidation & 1  & - & Foundation    & 3  & - & - & - & - & - & - \\
	\cmidrule{2-12}
	\multirow{3}{*}{MAE}
		& Expansion     & 10 & - & Consolidation & 2  & - & - & - & - & - & - \\
		& Expansion     & 10 & - & Foundation    & 2  & - & - & - & - & - & - \\
		& Consolidation & 2  & - & Foundation    & 2  & - & - & - & - & 

# Other statistics